In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
df = pd.read_csv("../data/raw/payflow_transactions.csv")
print(df.shape)
df.head()

(150000, 20)


,transaction_id,timestamp,user_id,merchant_id,amount,currency,merchant_category,payment_method,entry_mode,account_age_days,user_home_country,kyc_verified,device_type,is_new_device,ip_country,is_vpn_or_proxy,distance_from_home_km,failed_attempts_1h,is_international,is_fraud
0,TX_000001,2026-05-01 00:04:59,USR_04153,MER_375,154.51,SAR,electronics,credit_card,online_no_3ds,452.0,SA,0,ios,0,SA,0,24.10,0,0,0
1,TX_000002,2026-05-01 00:23:03,USR_07866,MER_261,577.81,SAR,electronics,credit_card,chip,766.0,SA,1,pos,0,NaN,0,45.27,1,0,0
2,TX_000003,2026-05-01 00:23:58,USR_09294,MER_465,528.23,EGP,groceries,credit_card,contactless,228.0,EG,1,pos,0,NaN,0,19.25,0,0,0
3,TX_000004,2026-05-01 00:26:20,USR_06746,MER_152,1415.34,EGP,groceries,credit_card,online_3ds,545.0,EG,1,android,0,EG,0,15.72,1,0,0
4,TX_000005,2026-05-01 00:28:11,USR_02578,MER_396,87.08,USD,fashion,debit_card,contactless,318.0,US,0,pos,0,NaN,0,7.79,0,0,0


In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

In [4]:
ratio = df['is_fraud'].value_counts(normalize=True)

print(f"Non-fraud: {ratio[0]:.2%}")
print(f"Fraud: {ratio[1]:.2%}")

Non-fraud: 97.99%
Fraud: 2.01%


In [5]:
monthly = (
    df.groupby(df['timestamp'].dt.to_period('M'))['is_fraud']
      .agg(fraud_rate='mean', n_fraud='sum')
)
monthly['fraud_rate'] = (monthly['fraud_rate'] * 100).round(2)
monthly

,fraud_rate,n_fraud
timestamp,,
2026-05,2.17,563
2026-06,1.95,489
2026-07,2.04,527
2026-08,1.95,504
2026-09,1.68,418
2026-10,2.26,508


In [6]:
print(f"dup: {df.duplicated().sum()}")
missing = df.isnull().sum()
missing[missing > 0]

dup: 0


account_age_days          1493
device_type               2231
ip_country               17561
distance_from_home_km     9158
dtype: int64

In [7]:
fx_to_usd = {
     "USD": 1.0,
         "EUR": 0.92,
         "GBP": 0.79,
         "SAR": 3.75,
         "EGP": 48.5,
}

df["amount_usd"] = (df["amount"] / df["currency"].map(fx_to_usd)).round(2)

In [8]:
df["hour_of_day"] = df["timestamp"].dt.hour
df["is_night"] = df["timestamp"].dt.hour.between(1, 5).astype(int)
df["ip_country_mismatch"] = ((df["ip_country"].notna()) & (df["ip_country"] != df["user_home_country"])).astype(int)

In [9]:
df[["timestamp", "amount_usd"]].describe().round(2).T

,count,mean,min,25%,50%,75%,max,std
timestamp,150000,2026-07-30 00:05:27.116093440,2026-05-01 00:04:59,2026-06-14 19:22:51,2026-07-29 20:19:02.500000,2026-09-12 21:51:30.249999872,2026-10-27 23:58:40,NaN
amount_usd,150000.0,110.64,1.0,21.34,48.0,122.44,5893.81,183.71


In [10]:
df = df.drop(columns=["transaction_id", "user_id", "merchant_id", "amount"])

In [11]:
start = df["timestamp"].min().normalize()

train_end = start + pd.Timedelta(days=120)
val_end = train_end + pd.Timedelta(days=30)
test_end = val_end + pd.Timedelta(days=30)

In [12]:
train = df[df['timestamp'] < train_end].copy()
val = df[(df['timestamp'] >= train_end) & (df['timestamp'] < val_end)].copy()
test = df[(df['timestamp'] >= val_end) & (df['timestamp'] < test_end)].copy()

for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f"{name}: {len(part):>7} rows | "
          f"{part['timestamp'].min().date()} -> {part['timestamp'].max().date()} | "
          f"fraud rate: {part['is_fraud'].mean():.2%}")

train:  100177 rows | 2026-05-01 -> 2026-08-28 | fraud rate: 2.05%
val:   24720 rows | 2026-08-29 -> 2026-09-27 | fraud rate: 1.58%
test:   25103 rows | 2026-09-28 -> 2026-10-27 | fraud rate: 2.26%


In [13]:
train = train.drop(columns=["timestamp"])
val = val.drop(columns=["timestamp"])
test = test.drop(columns=["timestamp"])

In [14]:
x_train = train.drop(columns=["is_fraud"])
y_train = train["is_fraud"]

In [15]:
x_val = val.drop(columns=["is_fraud"])
y_val = val["is_fraud"]

In [16]:
NUMERIC_COLS: list[str] = [
     "account_age_days",
     "failed_attempts_1h",
     "hour_of_day",
]

In [17]:
LOG_NUMERIC_COLS: list[str] = [
    "amount_usd",
    "distance_from_home_km",
]

In [18]:
BINARY_COLS: list[str] = [
     "kyc_verified",
     "is_new_device",
     "is_vpn_or_proxy",
     "is_international",
     "is_night",
     "ip_country_mismatch"
]

In [19]:
CATEGORICAL_COLS: list[str] = [
     "currency",
     "merchant_category",
     "payment_method",
     "entry_mode",
     "user_home_country",
     "device_type",
     "ip_country",
]


In [20]:
num_pipeline = Pipeline(
     steps=[
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
     ]
)

In [21]:
log_num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

In [22]:
cat_pipeline = Pipeline(
     steps=[
          ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
     ]
)

In [23]:
preprocessor = ColumnTransformer(
     transformers= [
          ("num", num_pipeline, NUMERIC_COLS),
          ("log_num", log_num_pipeline, LOG_NUMERIC_COLS),
          ("cat", cat_pipeline, CATEGORICAL_COLS),
          ("bin", "passthrough", BINARY_COLS)
     ]
)

In [24]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = [
     ("logistic", LogisticRegression(
          penalty="l2",
          C=1.0,
          solver="liblinear",
          class_weight="balanced",
          random_state=42,
          n_jobs=1
     )),
     ("random_forest", RandomForestClassifier(
          n_estimators=400,
          max_depth=10,
          min_samples_split=5,
          min_samples_leaf=2,
          max_features="sqrt",
          class_weight="balanced",
          random_state=42,
          n_jobs=1
     )),
     ("xgboost",
      XGBClassifier(
           n_estimators=400,
           learning_rate=0.03,
           max_depth=4,
           min_child_weight=5,
           subsample=0.8,
           colsample_bytree=0.8,
           reg_alpha=0.5,
           reg_lambda=2.0,
           eval_metric="logloss",
           scale_pos_weight=scale_pos_weight,
           random_state=42,
           n_jobs=1,
     )),
     ("lightgbm", LGBMClassifier(
          n_estimators=400,
          learning_rate=0.03,
          max_depth=4,
          num_leaves=15, 
          min_child_samples=20,
          subsample=0.8,
          subsample_freq=1,
          colsample_bytree=0.8,
          reg_alpha=0.5,
          reg_lambda=2.0,
          scale_pos_weight=scale_pos_weight,
          random_state=42,
          n_jobs=1,
          verbose=-1,              
     ))
]

In [25]:
val_results = []

for name, model in models:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipe.fit(x_train, y_train)

    train_probs = pipe.predict_proba(x_train)[:, 1]
    val_probs = pipe.predict_proba(x_val)[:, 1]

    val_results.append({
        "Model": name,
        "Train PR-AUC": round(average_precision_score(y_train, train_probs), 4),
        "Val PR-AUC": round(average_precision_score(y_val, val_probs), 4),
        "Train ROC-AUC": round(roc_auc_score(y_train, train_probs), 4),
        "Val ROC-AUC": round(roc_auc_score(y_val, val_probs), 4),
    })

pd.DataFrame(val_results).sort_values("Val PR-AUC", ascending=False)

,Model,Train PR-AUC,Val PR-AUC,Train ROC-AUC,Val ROC-AUC
3,lightgbm,0.5753,0.4134,0.9530,0.9091
2,xgboost,0.5730,0.4118,0.9529,0.9088
0,logistic,0.4783,0.3871,0.9247,0.9096
1,random_forest,0.6810,0.3575,0.9620,0.9004
